# Assignment 09: Join and Merge in SQL (SQLite Version)

### Due 12 November 2025

### Introduction

For this assignment, you will continue working with SQL databases using SQLite. You should use Python to write the SQL queries. If possible, please submit your answers in PDF format. The data and questions are listed below.

In [28]:
import sqlite3
import pandas as pd

# Create in-memory database
conn = sqlite3.connect(':memory:')

# Create tables
conn.execute('''
CREATE TABLE directors (
    director_id INTEGER PRIMARY KEY AUTOINCREMENT,
    director_name TEXT,
    country TEXT,
    birth_year INTEGER,
    awards INTEGER
)''')

conn.execute('''
CREATE TABLE movies (
    movie_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    director_id INTEGER,
    release_year INTEGER,
    box_office REAL,
    rating REAL,
    FOREIGN KEY (director_id) REFERENCES directors(director_id)
)''')

# Insert data
directors_data = [
    ('Christopher Nolan', 'UK', 1970, 5),
    ('Greta Gerwig', 'USA', 1983, 3),
    ('Bong Joon-ho', 'South Korea', 1969, 4),
    ('Sofia Coppola', 'USA', 1971, 2),
    ('Pedro Almodóvar', 'Spain', 1949, 6),
    ('Agnès Varda', 'France', 1928, 4)
]
conn.executemany('INSERT INTO directors (director_name, country, birth_year, awards) VALUES (?,?,?,?)', directors_data)

movies_data = [
    ('Oppenheimer', 1, 2023, 950000000.00, 8.5),
    ('Barbie', 2, 2023, 1440000000.00, 7.0),
    ('Parasite', 3, 2019, 258773645.00, 8.9),
    ('Lost in Translation', 4, 2003, 119723856.00, 7.7),
    ('Pain and Glory', 5, 2019, 38219573.00, 7.5),
    ('Faces Places', 6, 2017, 903996.00, 7.9),
    ('Inception', 1, 2010, 836836967.00, 8.8),
    ('Lady Bird', 2, 2017, 78965367.00, 7.4)
]
conn.executemany('''
    INSERT INTO movies (title, director_id, release_year, box_office, rating)
    VALUES (?,?,?,?,?)''', movies_data)
conn.commit()

In [29]:
# Read and print the movies table
df_movies = pd.read_sql_query("SELECT * FROM movies", conn)
print(df_movies.to_string(index=False))

 movie_id               title  director_id  release_year   box_office  rating
        1         Oppenheimer            1          2023  950000000.0     8.5
        2              Barbie            2          2023 1440000000.0     7.0
        3            Parasite            3          2019  258773645.0     8.9
        4 Lost in Translation            4          2003  119723856.0     7.7
        5      Pain and Glory            5          2019   38219573.0     7.5
        6        Faces Places            6          2017     903996.0     7.9
        7           Inception            1          2010  836836967.0     8.8
        8           Lady Bird            2          2017   78965367.0     7.4


1. Write a query using `INNER JOIN` to display the movie title, director name, and box office earnings for all movies, ordered by box office earnings in descending order

In [30]:
query = """
SELECT m.title, d.director_name, m.box_office
FROM movies m
INNER JOIN directors d ON m.director_id = d.director_id
ORDER BY m.box_office DESC;
"""
df_task1 = pd.read_sql_query(query, conn)
print(df_task1.to_string(index=False))

              title     director_name   box_office
             Barbie      Greta Gerwig 1440000000.0
        Oppenheimer Christopher Nolan  950000000.0
          Inception Christopher Nolan  836836967.0
           Parasite      Bong Joon-ho  258773645.0
Lost in Translation     Sofia Coppola  119723856.0
          Lady Bird      Greta Gerwig   78965367.0
     Pain and Glory   Pedro Almodóvar   38219573.0
       Faces Places       Agnès Varda     903996.0


2. Using a `LEFT JOIN`, find all directors and count the number of movies they have directed.

In [31]:
query_task2 = '''
SELECT d.director_name, COUNT(m.movie_id) AS movie_count
FROM directors d
LEFT JOIN movies m ON d.director_id = m.director_id
GROUP BY d.director_id, d.director_name
ORDER BY movie_count DESC, d.director_name;
'''
df_task2 = pd.read_sql_query(query_task2, conn)
print(df_task2.to_string(index=False))

    director_name  movie_count
Christopher Nolan            2
     Greta Gerwig            2
      Agnès Varda            1
     Bong Joon-ho            1
  Pedro Almodóvar            1
    Sofia Coppola            1


3. Write a `SELF JOIN` query to compare the ratings of movies by the same director. Show only pairs where the second movie has a higher rating than the first.

In [32]:
query_task3 = '''
SELECT m1.title AS movie1, m2.title AS movie2, d.director_name, m1.rating AS rating1, m2.rating AS rating2
FROM movies m1
JOIN movies m2 ON m1.director_id = m2.director_id AND m1.movie_id <> m2.movie_id AND m2.rating > m1.rating
JOIN directors d ON m1.director_id = d.director_id
ORDER BY d.director_name, rating1, rating2;
'''
df_task3 = pd.read_sql_query(query_task3, conn)
print(df_task3.to_string(index=False))

     movie1    movie2     director_name  rating1  rating2
Oppenheimer Inception Christopher Nolan      8.5      8.8
     Barbie Lady Bird      Greta Gerwig      7.0      7.4


4. Using appropriate joins, find directors who have made movies with above-average box office earnings (compared to all movies in the database).

In [33]:
query_task4 = '''
SELECT DISTINCT d.director_name
FROM directors d
JOIN movies m ON d.director_id = m.director_id
WHERE m.box_office > (SELECT AVG(box_office) FROM movies)
ORDER BY d.director_name;
'''
df_task4 = pd.read_sql_query(query_task4, conn)
print(df_task4.to_string(index=False))

    director_name
Christopher Nolan
     Greta Gerwig


5. Create a query using `CROSS JOIN` to show all possible combinations of directors and movies, even if they did not direct them. Limit the output to 10 rows.

In [34]:
query_task5 = '''
SELECT d.director_name, m.title
FROM directors d
CROSS JOIN movies m
LIMIT 10;
'''
df_task5 = pd.read_sql_query(query_task5, conn)
print(df_task5.to_string(index=False))

    director_name               title
Christopher Nolan         Oppenheimer
Christopher Nolan              Barbie
Christopher Nolan            Parasite
Christopher Nolan Lost in Translation
Christopher Nolan      Pain and Glory
Christopher Nolan        Faces Places
Christopher Nolan           Inception
Christopher Nolan           Lady Bird
     Greta Gerwig         Oppenheimer
     Greta Gerwig              Barbie


6. Write a query that uses `UNION` to create a list of all director names and movie titles in a single column. Label the column `name` and include a column (called `type`) indicating if it is a director or movie. Order the results by type and name.

In [35]:
query_task6 = '''
SELECT director_name AS name, 'director' AS type FROM directors
UNION
SELECT title AS name, 'movie' AS type FROM movies
ORDER BY type, name;
'''
df_task6 = pd.read_sql_query(query_task6, conn)
print(df_task6.to_string(index=False))

               name     type
        Agnès Varda director
       Bong Joon-ho director
  Christopher Nolan director
       Greta Gerwig director
    Pedro Almodóvar director
      Sofia Coppola director
             Barbie    movie
       Faces Places    movie
          Inception    movie
          Lady Bird    movie
Lost in Translation    movie
        Oppenheimer    movie
     Pain and Glory    movie
           Parasite    movie


7. Using appropriate joins, find the director with the highest average movie rating. Show only the row with the director's name, average rating, and number of movies.

In [36]:
query_task7 = '''
SELECT d.director_name, AVG(m.rating) AS avg_rating, COUNT(m.movie_id) AS num_movies
FROM directors d
JOIN movies m ON d.director_id = m.director_id
GROUP BY d.director_id, d.director_name
ORDER BY avg_rating DESC
LIMIT 1;
'''
df_task7 = pd.read_sql_query(query_task7, conn)
print(df_task7.to_string(index=False))

director_name  avg_rating  num_movies
 Bong Joon-ho         8.9           1


8. Create a query using `LEFT JOIN` and `IS NULL` to find whether there are directors who have not directed any movies.

In [37]:
query_task8 = '''
SELECT d.director_name
FROM directors d
LEFT JOIN movies m ON d.director_id = m.director_id
WHERE m.movie_id IS NULL;
'''
df_task8 = pd.read_sql_query(query_task8, conn)
print(df_task8.to_string(index=False))

print("Answer: There's no director who hasn't directed any movies in the data.")

Empty DataFrame
Columns: [director_name]
Index: []
Answer: There's no director who hasn't directed any movies in the data.


9. Using appropriate joins, find pairs of movies released in the same year, along with their directors' names. Please do not match a movie with itself.

In [38]:
query_task9 = '''
SELECT m1.title AS movie1, d1.director_name AS director1,
       m2.title AS movie2, d2.director_name AS director2, m1.release_year
FROM movies m1
JOIN movies m2 ON m1.release_year = m2.release_year AND m1.movie_id <> m2.movie_id
JOIN directors d1 ON m1.director_id = d1.director_id
JOIN directors d2 ON m2.director_id = d2.director_id
ORDER BY m1.release_year, movie1, movie2;
'''
df_task9 = pd.read_sql_query(query_task9, conn)
print(df_task9.to_string(index=False))

        movie1         director1         movie2         director2  release_year
  Faces Places       Agnès Varda      Lady Bird      Greta Gerwig          2017
     Lady Bird      Greta Gerwig   Faces Places       Agnès Varda          2017
Pain and Glory   Pedro Almodóvar       Parasite      Bong Joon-ho          2019
      Parasite      Bong Joon-ho Pain and Glory   Pedro Almodóvar          2019
        Barbie      Greta Gerwig    Oppenheimer Christopher Nolan          2023
   Oppenheimer Christopher Nolan         Barbie      Greta Gerwig          2023


10. Show the age of each director when they released their movies. Create a column entitled `age_at_release` in your output. Order the results by the director's name and the movie's release year.

In [39]:
query_task10 = '''
SELECT d.director_name, m.title, m.release_year, (m.release_year - d.birth_year) AS age_at_release
FROM directors d
JOIN movies m ON d.director_id = m.director_id
ORDER BY d.director_name, m.release_year;
'''
df_task10 = pd.read_sql_query(query_task10, conn)
print(df_task10.to_string(index=False))

    director_name               title  release_year  age_at_release
      Agnès Varda        Faces Places          2017              89
     Bong Joon-ho            Parasite          2019              50
Christopher Nolan           Inception          2010              40
Christopher Nolan         Oppenheimer          2023              53
     Greta Gerwig           Lady Bird          2017              34
     Greta Gerwig              Barbie          2023              40
  Pedro Almodóvar      Pain and Glory          2019              70
    Sofia Coppola Lost in Translation          2003              32


Good luck! 😃